# Wisconsin 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Wisconsin, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals).

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `rep_general_total`, `dem_general_total`, `lib_general_total`, `wgr_general_total`, `ind_general_total`

**Last Updated**: 2025/10/11

## 0. Library Import

In [12]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [13]:
# WI 2008 dataset path
PRIMARY_PATH1 = r"../../data/raw/2008/WI/20080219__wi__primary__ward.csv"
PRIMARY_PATH2 = r"../../data/raw/2008/WI/20080909__wi__primary__ward.csv"

GENERAL_PATH1 = r"../../data/raw/2008/WI/20080401__wi__general__ward.csv"
GENERAL_PATH2 = r"../../data/raw/2008/WI/20081104__wi__general__ward.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/WI/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

Inteestingly, from OpenElections GitHub for Wisconsin, there are two files each for primary and general. Thus, we will process each CSV file independently, then merge them together later.

### a. Primary Election Dataset

In [14]:
# Load primary data 1
primary_df1 = pd.read_csv(PRIMARY_PATH1)
primary_df1.head(DISPLAY_ROWS)

,county,ward,office,district,total votes,party,candidate,votes
0,Chippewa,Town Of Anson Wards 1 - 3,"Chippewa County Circuit Court, Branch 3",NaN,493,NP,Robert A. Ferg,119
1,Chippewa,Town Of Anson Wards 1 - 3,"Chippewa County Circuit Court, Branch 3",NaN,493,NP,Julie Anderl,200
2,Chippewa,Town Of Anson Wards 1 - 3,"Chippewa County Circuit Court, Branch 3",NaN,493,NP,Steven R. Cray,174
3,Chippewa,Town Of Anson Wards 1 - 3,"Chippewa County Circuit Court, Branch 3",NaN,493,NaN,Scattering,0
4,Chippewa,Town Of Arthur,"Chippewa County Circuit Court, Branch 3",NaN,150,NP,Robert A. Ferg,38
5,Chippewa,Town Of Arthur,"Chippewa County Circuit Court, Branch 3",NaN,150,NP,Julie Anderl,76
6,Chippewa,Town Of Arthur,"Chippewa County Circuit Court, Branch 3",NaN,150,NP,Steven R. Cray,36
7,Chippewa,Town Of Arthur,"Chippewa County Circuit Court, Branch 3",NaN,150,NaN,Scattering,0
8,Chippewa,Town Of Auburn,"Chippewa County Circuit Court, Branch 3",NaN,157,NP,Robert A. Ferg,32
9,Chippewa,Town Of Auburn,"Chippewa County Circuit Court, Branch 3",NaN,157,NP,Julie Anderl,87


In [15]:
# Different values in 'office' column
primary_df1["office"].value_counts()

office
President                                   70720
Outagamie County Circuit Court, Branch 2      364
St. Croix County Circuit Court, Branch 4      228
Chippewa County Circuit Court, Branch 3       152
Florence-Forest County Circuit Court          104
Name: count, dtype: int64

In [16]:
# Only keep rows where 'office' is 'President'
primary_df1 = primary_df1[primary_df1["office"] == "President"]
primary_df1.shape

(70720, 8)

In [17]:
# Now, drop the "office" column as it's no longer needed
primary_df1 = primary_df1.drop(columns="office").reset_index(drop=True)
primary_df1.head(DISPLAY_ROWS)

,county,ward,district,total votes,party,candidate,votes
0,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Dennis Kucinich,1
1,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Hillary Clinton,119
2,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Joe Biden,0
3,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Mike Gravel,0
4,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Chris Dodd,0
5,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Barack Obama,118
6,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,John Edwards,3
7,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Bill Richardson,0
8,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Uninstructed Delegation,0
9,Adams,Town Of Adams Wards 1 & 2,NaN,241,DEM,Scattering,0


In [18]:
# Missing values in each column
primary_df1.isna().sum()

county             0
ward               0
district       70720
total votes        0
party              0
candidate          0
votes              0
dtype: int64

This shows that there are no values in `district` column. Thus, we can also drop this.

In [19]:
# Drop the 'district' column
primary_df1 = primary_df1.drop(columns="district").reset_index(drop=True)
primary_df1.head(DISPLAY_ROWS)

,county,ward,total votes,party,candidate,votes
0,Adams,Town Of Adams Wards 1 & 2,241,DEM,Dennis Kucinich,1
1,Adams,Town Of Adams Wards 1 & 2,241,DEM,Hillary Clinton,119
2,Adams,Town Of Adams Wards 1 & 2,241,DEM,Joe Biden,0
3,Adams,Town Of Adams Wards 1 & 2,241,DEM,Mike Gravel,0
4,Adams,Town Of Adams Wards 1 & 2,241,DEM,Chris Dodd,0
5,Adams,Town Of Adams Wards 1 & 2,241,DEM,Barack Obama,118
6,Adams,Town Of Adams Wards 1 & 2,241,DEM,John Edwards,3
7,Adams,Town Of Adams Wards 1 & 2,241,DEM,Bill Richardson,0
8,Adams,Town Of Adams Wards 1 & 2,241,DEM,Uninstructed Delegation,0
9,Adams,Town Of Adams Wards 1 & 2,241,DEM,Scattering,0


We want presidential election data on county-level, not ward-level. Thus, we will group the vote counts by county and drop the `ward` column.

I'm quite unsure about what the `total votes` column here represent since all rows in the snippet have the same count, so maybe I will aggregate the count instead of using that `total votes` column. My guess is that it represents the total votes by ward, regardless of candidate. I will drop this column so there will be less confusion.

In [20]:
# Drop the 'total votes' column
primary_df1 = primary_df1.drop(columns="total votes").reset_index(drop=True)

# Make sure votes are numeric
primary_df1["votes"] = pd.to_numeric(primary_df1["votes"], errors="coerce").fillna(0)

# Aggregate ward vote couns into county vote counts
primary_df1 = (
    primary_df1.
    groupby(["county", "party", "candidate"], as_index=False)["votes"]
    .sum()
)[["county", "candidate", "party", "votes"]]        # Reorder columns

# Snippet at the aggregated data
primary_df1.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Adams,Barack Obama,DEM,1782
1,Adams,Bill Richardson,DEM,2
2,Adams,Chris Dodd,DEM,1
3,Adams,Dennis Kucinich,DEM,7
4,Adams,Hillary Clinton,DEM,2104
5,Adams,Joe Biden,DEM,3
6,Adams,John Edwards,DEM,48
7,Adams,Mike Gravel,DEM,3
8,Adams,Scattering,DEM,3
9,Adams,Uninstructed Delegation,DEM,2


In [22]:
# Unique parties in primary1_df
primary_df1["party"].value_counts()

party
DEM    720
REP    720
Name: count, dtype: int64

In [23]:
# Candidates in primary1_df
primary_df1["candidate"].value_counts()

candidate
Uninstructed Delegation    144
Scattering                 144
Duncan Hunter               72
Rudy Giuliani               72
Ron Paul                    72
Mitt Romney                 72
Mike Huckabee               72
John McCain                 72
Fred Thompson               72
Barack Obama                72
Bill Richardson             72
Mike Gravel                 72
John Edwards                72
Joe Biden                   72
Hillary Clinton             72
Dennis Kucinich             72
Chris Dodd                  72
Tom Tancredo                72
Name: count, dtype: int64

In [25]:
# Final look at the (supposed) cleaned primary1_df
primary_df1.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Adams,Barack Obama,DEM,1782
1,Adams,Bill Richardson,DEM,2
2,Adams,Chris Dodd,DEM,1
3,Adams,Dennis Kucinich,DEM,7
4,Adams,Hillary Clinton,DEM,2104
5,Adams,Joe Biden,DEM,3
6,Adams,John Edwards,DEM,48
7,Adams,Mike Gravel,DEM,3
8,Adams,Scattering,DEM,3
9,Adams,Uninstructed Delegation,DEM,2


In [27]:
# Shape after preprocessing
primary_df1.shape

(1440, 4)

Finishing up with this dataset, we now look at the other primary election dataset.

In [28]:
# Load primary data
primary_df2 = pd.read_csv(PRIMARY_PATH2)
primary_df2.head(DISPLAY_ROWS)

,county,ward,office,district,total votes,party,candidate,votes
0,Brown,Town Of Lawrence Wards 1 - 3,State Senate,2,1,DEM,Scattering,1
1,Brown,Village Of Allouez Wards 1 & 2,State Senate,2,3,DEM,Scattering,3
2,Brown,Village Of Allouez Wards 3 & 4,State Senate,2,6,DEM,Scattering,6
3,Brown,Village Of Allouez Wards 5 & 6,State Senate,2,7,DEM,Scattering,7
4,Brown,Village Of Allouez Wards 7 - 9,State Senate,2,0,DEM,Scattering,0
5,Brown,Village Of Ashwaubenon Wards 1 & 2,State Senate,2,6,DEM,Scattering,6
6,Brown,Village Of Ashwaubenon Wards 3 & 4,State Senate,2,3,DEM,Scattering,3
7,Brown,Village Of Ashwaubenon Wards 5 & 6,State Senate,2,1,DEM,Scattering,1
8,Brown,Village Of Ashwaubenon Wards 7 & 8,State Senate,2,0,DEM,Scattering,0
9,Brown,Village Of Ashwaubenon Ward 9,State Senate,2,2,DEM,Scattering,2


In [30]:
# Different values in 'office' column
primary_df2["office"].value_counts()

office
House             20217
State Assembly    18232
State Senate       6735
Name: count, dtype: int64

Oh, there are no presidential data in this dataframe. Then, we can just safely ignore it and proceed with general dataset.

In [41]:
# Rename for ease of future steps
primary_df = primary_df1

### b. General Election Dataset

In [32]:
# Load general data
general_df1 = pd.read_csv(GENERAL_PATH1)
general_df1.head(DISPLAY_ROWS)

,county,ward,office,district,total votes,party,candidate,votes
0,Barron,Town Of Almena Wards 1 - 3,"Barron County Circuit Court, Branch 2",NaN,81,NP,Timothy M. Doyle,81
1,Barron,Town Of Almena Wards 1 - 3,"Barron County Circuit Court, Branch 2",NaN,81,NaN,Scattering,0
2,Barron,Town Of Arland,"Barron County Circuit Court, Branch 2",NaN,42,NP,Timothy M. Doyle,42
3,Barron,Town Of Arland,"Barron County Circuit Court, Branch 2",NaN,42,NaN,Scattering,0
4,Barron,Town Of Barron Wards 1 & 2,"Barron County Circuit Court, Branch 2",NaN,110,NP,Timothy M. Doyle,110
5,Barron,Town Of Barron Wards 1 & 2,"Barron County Circuit Court, Branch 2",NaN,110,NaN,Scattering,0
6,Barron,Town Of Bear Lake,"Barron County Circuit Court, Branch 2",NaN,141,NP,Timothy M. Doyle,141
7,Barron,Town Of Bear Lake,"Barron County Circuit Court, Branch 2",NaN,141,NaN,Scattering,0
8,Barron,Town Of Cedar Lake,"Barron County Circuit Court, Branch 2",NaN,185,NP,Timothy M. Doyle,185
9,Barron,Town Of Cedar Lake,"Barron County Circuit Court, Branch 2",NaN,185,NaN,Scattering,0


In [35]:
# Different values in 'office' column
general_df1["office"].value_counts()

office
Supreme Court                                       11553
Court of Appeals                                     5831
Milwaukee County Circuit Court, Branch 40            1395
Milwaukee County Circuit Court, Branch 17             930
Milwaukee County Circuit Court, Branch 41             930
Milwaukee County Circuit Court, Branch 32             930
Milwaukee County Circuit Court, Branch 31             930
Milwaukee County Circuit Court, Branch 21             930
Milwaukee County Circuit Court, Branch 27             930
Dane County Circuit Court, Branch 1                   446
Dane County Circuit Court, Branch 7                   446
Waukesha County Circuit Court, Branch 2               412
Waukesha County Circuit Court, Branch 5               412
Waukesha County Circuit Court, Branch 6               412
Outagamie County Circuit Court, Branch 3              360
Outagamie County Circuit Court, Branch 2              360
Marathon County Circuit Court, Branch 3               272
Kenosha

There are a lot of different values in the `office` column. We might want to check if there are any of them include presidential election information.

In [38]:
# Does any 'office' entry mention president?
has_pres = general_df1["office"].astype(str).str.contains(r"\bpresident\b", case=False, na=False).any()
has_pres


False

Then, there is no presidential data in this dataset. Thus, we can move on to process the other CSV file.

In [39]:
# Load general data
general_df2 = pd.read_csv(GENERAL_PATH2)
general_df2.head(DISPLAY_ROWS)

,county,ward,office,district,total votes,party,candidate,votes
0,Adams,Town Of Adams Wards 1 & 2,Adams County District Attorney,NaN,481,DEM,Mark D. Thibodeau,475
1,Adams,Town Of Adams Wards 1 & 2,Adams County District Attorney,NaN,481,NaN,Scattering,6
2,Adams,Town Of Big Flats,Adams County District Attorney,NaN,390,DEM,Mark D. Thibodeau,387
3,Adams,Town Of Big Flats,Adams County District Attorney,NaN,390,NaN,Scattering,3
4,Adams,Town Of Colburn,Adams County District Attorney,NaN,96,DEM,Mark D. Thibodeau,96
5,Adams,Town Of Colburn,Adams County District Attorney,NaN,96,NaN,Scattering,0
6,Adams,Town Of Dell Prairie Wards 1 & 2,Adams County District Attorney,NaN,517,DEM,Mark D. Thibodeau,514
7,Adams,Town Of Dell Prairie Wards 1 & 2,Adams County District Attorney,NaN,517,NaN,Scattering,3
8,Adams,Town Of Easton Wards 1 & 2,Adams County District Attorney,NaN,338,DEM,Mark D. Thibodeau,334
9,Adams,Town Of Easton Wards 1 & 2,Adams County District Attorney,NaN,338,NaN,Scattering,4


In [40]:
# Different values in 'office' column
general_df2["office"].value_counts()

office
President                              36000
House                                  11726
State Assembly                         10335
State Senate                            4142
Milwaukee County District Attorney       924
                                       ...  
Green Lake County District Attorney       32
Vilas County District Attorney            30
Kewaunee County District Attorney         28
Pepin County District Attorney            22
Florence County District Attorney         16
Name: count, Length: 75, dtype: int64

Now, there are information about presidential election in this dataset. Then, we can first filter rows with "President" value in `office` column and move on from there.

In [43]:
# Only keep rows where 'office' is 'President'
general_df = general_df2[general_df2["office"] == "President"]      # Rename to general_df also
general_df.shape

(36000, 8)

In [44]:
# Now, drop the "office" column as it's no longer needed
general_df = general_df.drop(columns="office").reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,ward,district,total votes,party,candidate,votes
0,Adams,Town Of Adams Wards 1 & 2,NaN,663,DEM,Joe Biden Barack Obama,405
1,Adams,Town Of Adams Wards 1 & 2,NaN,663,REP,Sarah Palin John McCain,235
2,Adams,Town Of Adams Wards 1 & 2,NaN,663,WGR,Rosa Clemente Cynthia McKinney,1
3,Adams,Town Of Adams Wards 1 & 2,NaN,663,LIB,Wayne A. Root Bob Barr,4
4,Adams,Town Of Adams Wards 1 & 2,NaN,663,IND,Stewart A. Alexander Brian Moore,0
5,Adams,Town Of Adams Wards 1 & 2,NaN,663,IND,Robert Moses Gloria LaRiva,0
6,Adams,Town Of Adams Wards 1 & 2,NaN,663,IND,Matt Gonzalez Ralph Nader,7
7,Adams,Town Of Adams Wards 1 & 2,NaN,663,IND,Darrell L. Castle Chuck Baldwin,6
8,Adams,Town Of Adams Wards 1 & 2,NaN,663,IND,David J. Klimisch Jeffrey J. Wamboldt,2
9,Adams,Town Of Adams Wards 1 & 2,NaN,663,NaN,Scattering,3


In [46]:
# Missing values in each column
general_df.isna().sum()

county             0
ward               0
district       36000
total votes        0
party           3600
candidate          0
votes              0
dtype: int64

This shows that there are no values in `district` column. Thus, we can also drop this.

In [47]:
# Drop the 'district' column
general_df = general_df.drop(columns="district").reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,ward,total votes,party,candidate,votes
0,Adams,Town Of Adams Wards 1 & 2,663,DEM,Joe Biden Barack Obama,405
1,Adams,Town Of Adams Wards 1 & 2,663,REP,Sarah Palin John McCain,235
2,Adams,Town Of Adams Wards 1 & 2,663,WGR,Rosa Clemente Cynthia McKinney,1
3,Adams,Town Of Adams Wards 1 & 2,663,LIB,Wayne A. Root Bob Barr,4
4,Adams,Town Of Adams Wards 1 & 2,663,IND,Stewart A. Alexander Brian Moore,0
5,Adams,Town Of Adams Wards 1 & 2,663,IND,Robert Moses Gloria LaRiva,0
6,Adams,Town Of Adams Wards 1 & 2,663,IND,Matt Gonzalez Ralph Nader,7
7,Adams,Town Of Adams Wards 1 & 2,663,IND,Darrell L. Castle Chuck Baldwin,6
8,Adams,Town Of Adams Wards 1 & 2,663,IND,David J. Klimisch Jeffrey J. Wamboldt,2
9,Adams,Town Of Adams Wards 1 & 2,663,NaN,Scattering,3


Now, we will aggregate the vote counts by counties.

In [48]:
# Drop the 'total votes' column
general_df = general_df.drop(columns="total votes").reset_index(drop=True)

# Make sure votes are numeric
general_df["votes"] = pd.to_numeric(general_df["votes"], errors="coerce").fillna(0)

# Aggregate ward vote couns into county vote counts
general_df = (
    general_df.
    groupby(["county", "party", "candidate"], as_index=False)["votes"]
    .sum()
)[["county", "candidate", "party", "votes"]]        # Reorder columns

# Snippet at the aggregated data
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Adams,Joe Biden Barack Obama,DEM,5806
1,Adams,Darrell L. Castle Chuck Baldwin,IND,43
2,Adams,David J. Klimisch Jeffrey J. Wamboldt,IND,6
3,Adams,Matt Gonzalez Ralph Nader,IND,76
4,Adams,Robert Moses Gloria LaRiva,IND,0
5,Adams,Stewart A. Alexander Brian Moore,IND,0
6,Adams,Wayne A. Root Bob Barr,LIB,27
7,Adams,Sarah Palin John McCain,REP,3974
8,Adams,Rosa Clemente Cynthia McKinney,WGR,16
9,Ashland,Joe Biden Barack Obama,DEM,5818


In [53]:
# Unique parties in general_df
general_df["party"].value_counts()

party
IND    360
DEM     72
LIB     72
REP     72
WGR     72
Name: count, dtype: int64

In [54]:
# Missing values in each column now
general_df.isna().sum()

county       0
candidate    0
party        0
votes        0
dtype: int64

In [55]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
Joe Biden Barack Obama                   72
Darrell L. Castle Chuck Baldwin          72
David J. Klimisch Jeffrey J. Wamboldt    72
Matt Gonzalez Ralph Nader                72
Robert Moses Gloria LaRiva               72
Stewart A. Alexander Brian Moore         72
Wayne A. Root Bob Barr                   72
Sarah Palin John McCain                  72
Rosa Clemente Cynthia McKinney           72
Name: count, dtype: int64

In [56]:
# Final look at the (supposed) cleaned primary1_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Adams,Joe Biden Barack Obama,DEM,5806
1,Adams,Darrell L. Castle Chuck Baldwin,IND,43
2,Adams,David J. Klimisch Jeffrey J. Wamboldt,IND,6
3,Adams,Matt Gonzalez Ralph Nader,IND,76
4,Adams,Robert Moses Gloria LaRiva,IND,0
5,Adams,Stewart A. Alexander Brian Moore,IND,0
6,Adams,Wayne A. Root Bob Barr,LIB,27
7,Adams,Sarah Palin John McCain,REP,3974
8,Adams,Rosa Clemente Cynthia McKinney,WGR,16
9,Ashland,Joe Biden Barack Obama,DEM,5818


In [57]:
# Shape after preprocessing
general_df.shape

(648, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: maps common forms (e.g., “Democratic”, “Republican”) to keys dem/rep so column names are stable
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [58]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: DEM -> dem, REP -> rep
    """
    return(s.str.lower())     

In [59]:
def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values

    # Return first token uppercase
    raw = str(name).strip()
    tokens = raw.split()
    return tokens[0].upper() if tokens else "UNKNOWN"

In [60]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [61]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_BARACK,pri_dem_BILL,pri_dem_CHRIS,pri_dem_DENNIS,pri_dem_HILLARY,pri_dem_JOE,pri_dem_JOHN,pri_dem_MIKE,pri_dem_SCATTERING,...,pri_rep_DUNCAN,pri_rep_FRED,pri_rep_JOHN,pri_rep_MIKE,pri_rep_MITT,pri_rep_RON,pri_rep_RUDY,pri_rep_SCATTERING,pri_rep_TOM,pri_rep_UNINSTRUCTED
0,Adams,1782,2,1,7,2104,3,48,3,3,...,4,12,725,605,26,97,5,0,0,5
1,Ashland,1800,2,1,13,1583,6,30,4,4,...,2,4,459,298,22,31,14,0,0,2
2,Barron,3970,1,2,26,3398,6,63,2,5,...,11,23,1422,1370,61,134,16,4,2,14
3,Bayfield,2157,1,2,15,1675,6,33,3,0,...,1,11,663,379,21,66,3,1,2,3
4,Brown,24737,18,23,70,18608,24,249,21,17,...,33,95,9862,6400,421,700,81,29,8,30
5,Buffalo,1163,1,3,7,1091,1,15,3,1,...,5,3,427,407,9,36,4,0,0,2
6,Burnett,1073,1,2,11,1266,2,29,1,0,...,3,14,575,401,16,83,5,4,1,3
7,Calumet,4648,4,2,13,3460,3,46,3,0,...,4,11,2103,1442,38,229,15,2,3,8
8,Chippewa,5519,4,1,11,4743,8,60,3,0,...,4,17,1733,2710,66,162,20,0,1,7
9,Clark,2751,2,3,10,2354,5,56,3,2,...,5,8,1040,1544,32,84,10,5,4,4


In [62]:
# Primary dataframe shape after pivot
primary_pivot.shape

(72, 21)

In [63]:
# Columns in primary_pivot
primary_pivot.columns

Index(['county', 'pri_dem_BARACK', 'pri_dem_BILL', 'pri_dem_CHRIS',
       'pri_dem_DENNIS', 'pri_dem_HILLARY', 'pri_dem_JOE', 'pri_dem_JOHN',
       'pri_dem_MIKE', 'pri_dem_SCATTERING', 'pri_dem_UNINSTRUCTED',
       'pri_rep_DUNCAN', 'pri_rep_FRED', 'pri_rep_JOHN', 'pri_rep_MIKE',
       'pri_rep_MITT', 'pri_rep_RON', 'pri_rep_RUDY', 'pri_rep_SCATTERING',
       'pri_rep_TOM', 'pri_rep_UNINSTRUCTED'],
      dtype='object')

A thing to notice here is that there are four columns represented "Uninstructed Delection" (`pri_rep_UNINSTRUCTED`/`pri_dem_UNINSTRUCTED`) and "Scattering" (`pri_rep_SCATTERING`/`pri_dem_SCATTERING`). Given there are still associated with DEM/REP, they will still affect the total count of the two parties. Thus, we will keep these columns a litle bit more to count the total votes by party per county, then drop it eventually at the end.

In [64]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_JOE,gen_ind_DARRELL,gen_ind_DAVID,gen_ind_MATT,gen_ind_ROBERT,gen_ind_STEWART,gen_lib_WAYNE,gen_rep_SARAH,gen_wgr_ROSA
0,Adams,5806,43,6,76,0,0,27,3974,16
1,Ashland,5818,11,2,59,0,1,15,2634,15
2,Barron,12078,46,4,140,3,3,72,10457,34
3,Bayfield,5972,13,4,65,1,0,15,3365,11
4,Brown,67269,174,15,638,16,11,426,55854,132
5,Buffalo,3949,17,1,64,2,0,25,2923,14
6,Burnett,4337,19,5,42,2,3,36,4200,24
7,Calumet,13295,76,11,196,0,3,83,12722,38
8,Chippewa,16239,67,5,283,2,4,55,13492,26
9,Clark,7454,48,11,151,3,3,45,6383,31


In [65]:
# General dataframe shape after pivot
general_pivot.shape

(72, 10)

In [67]:
# Columns in general_pivot
general_pivot.columns

Index(['county', 'gen_dem_JOE', 'gen_ind_DARRELL', 'gen_ind_DAVID',
       'gen_ind_MATT', 'gen_ind_ROBERT', 'gen_ind_STEWART', 'gen_lib_WAYNE',
       'gen_rep_SARAH', 'gen_wgr_ROSA'],
      dtype='object')

## 4. Merge Dataframes

Before merging, we verify that county names match across primary and general:

In [68]:
# Check if county names match between primary_df and general_df
primary_counties = set(primary_df["county"].unique())
general_counties = set(general_df["county"].unique())
common_counties = primary_counties.intersection(general_counties)
print(f"Number of common counties: {len(common_counties)} out of {len(primary_counties)}")

Number of common counties: 72 out of 72


Great. Since we know that all counties name are matched, we don't need to perform further data preprocessing to match the county names. Thus, we can now merge them:

In [69]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BARACK,pri_dem_BILL,pri_dem_CHRIS,pri_dem_DENNIS,pri_dem_HILLARY,pri_dem_JOE,pri_dem_JOHN,pri_dem_MIKE,pri_dem_SCATTERING,...,pri_rep_UNINSTRUCTED,gen_dem_JOE,gen_ind_DARRELL,gen_ind_DAVID,gen_ind_MATT,gen_ind_ROBERT,gen_ind_STEWART,gen_lib_WAYNE,gen_rep_SARAH,gen_wgr_ROSA
0,Adams,1782,2,1,7,2104,3,48,3,3,...,5,5806,43,6,76,0,0,27,3974,16
1,Ashland,1800,2,1,13,1583,6,30,4,4,...,2,5818,11,2,59,0,1,15,2634,15
2,Barron,3970,1,2,26,3398,6,63,2,5,...,14,12078,46,4,140,3,3,72,10457,34
3,Bayfield,2157,1,2,15,1675,6,33,3,0,...,3,5972,13,4,65,1,0,15,3365,11
4,Brown,24737,18,23,70,18608,24,249,21,17,...,30,67269,174,15,638,16,11,426,55854,132
5,Buffalo,1163,1,3,7,1091,1,15,3,1,...,2,3949,17,1,64,2,0,25,2923,14
6,Burnett,1073,1,2,11,1266,2,29,1,0,...,3,4337,19,5,42,2,3,36,4200,24
7,Calumet,4648,4,2,13,3460,3,46,3,0,...,8,13295,76,11,196,0,3,83,12722,38
8,Chippewa,5519,4,1,11,4743,8,60,3,0,...,7,16239,67,5,283,2,4,55,13492,26
9,Clark,2751,2,3,10,2354,5,56,3,2,...,4,7454,48,11,151,3,3,45,6383,31


In [70]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_BARACK,pri_dem_BILL,pri_dem_CHRIS,pri_dem_DENNIS,pri_dem_HILLARY,pri_dem_JOE,pri_dem_JOHN,pri_dem_MIKE,pri_dem_SCATTERING,pri_dem_UNINSTRUCTED,...,pri_rep_UNINSTRUCTED,gen_dem_JOE,gen_ind_DARRELL,gen_ind_DAVID,gen_ind_MATT,gen_ind_ROBERT,gen_ind_STEWART,gen_lib_WAYNE,gen_rep_SARAH,gen_wgr_ROSA
count,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,...,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000
mean,8984.041667,7.333333,6.958333,36.458333,6304.916667,10.486111,92.958333,7.180556,6.500000,11.958333,...,11.805556,23294.597222,70.444444,10.611111,244.513889,3.291667,7.500000,123.027778,17533.236111,58.555556
std,19272.973921,12.879725,15.452278,84.510177,10658.189758,19.211934,141.237432,15.019543,15.197294,21.428846,...,17.411862,44990.010224,86.962172,14.091807,346.390523,7.210004,18.259476,189.403212,26262.551924,87.741600
min,206.000000,0.000000,0.000000,1.000000,190.000000,0.000000,3.000000,0.000000,0.000000,0.000000,...,0.000000,1134.000000,0.000000,0.000000,4.000000,0.000000,0.000000,0.000000,185.000000,2.000000
25%,1803.000000,2.000000,1.000000,11.000000,1634.000000,3.000000,29.000000,1.750000,1.000000,3.000000,...,3.000000,5474.500000,21.750000,2.000000,68.000000,1.000000,1.000000,27.750000,4199.750000,16.750000
50%,3498.500000,4.000000,3.500000,16.500000,2999.500000,6.000000,48.500000,3.000000,2.500000,6.000000,...,6.000000,11035.500000,43.000000,5.500000,129.000000,1.000000,3.000000,62.500000,8911.500000,33.500000
75%,8534.750000,8.250000,6.250000,32.250000,6523.500000,10.250000,92.500000,6.500000,7.250000,14.000000,...,14.000000,21889.500000,81.250000,12.250000,309.500000,3.000000,8.000000,135.750000,21531.250000,63.750000
max,132501.000000,78.000000,109.000000,528.000000,73430.000000,143.000000,910.000000,95.000000,118.000000,156.000000,...,122.000000,319819.000000,540.000000,79.000000,2360.000000,56.000000,120.000000,1105.000000,149445.000000,589.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns

- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `lib_general_total` = sum of all `gen_lib_*` columns
    * `wgr_general_total` = sum of all `gen_wgr_*` columns
    * `ind_general_total` = sum of all `gen_ind_*` columns

In [71]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

In [72]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
lib_general_cols   = [c for c in merged_df.columns if c.startswith("gen_lib_")]
wgr_general_cols   = [c for c in merged_df.columns if c.startswith("gen_wgr_")]
ind_general_cols   = [c for c in merged_df.columns if c.startswith("gen_ing_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["lib_general_total"] = merged_df[lib_general_cols].sum(axis=1) if lib_general_cols else 0
merged_df["wgr_general_total"] = merged_df[wgr_general_cols].sum(axis=1) if wgr_general_cols else 0
merged_df["ind_general_total"] = merged_df[ind_general_cols].sum(axis=1) if ind_general_cols else 0

Now, we have added the total counts for all, we can safely drop the four columns we noted above since we will not need those for further analysis.

In [73]:
# Drop columns with UNRESTRICTED and SCATTERING data
merged_df = merged_df.drop(columns=["pri_rep_UNINSTRUCTED", "pri_dem_UNINSTRUCTED", "pri_rep_SCATTERING", "pri_dem_SCATTERING"]).reset_index(drop=True)
merged_df.shape

(72, 33)

In [75]:
# Print out all the column names in the final dataframe
merged_df.columns

Index(['county', 'pri_dem_BARACK', 'pri_dem_BILL', 'pri_dem_CHRIS',
       'pri_dem_DENNIS', 'pri_dem_HILLARY', 'pri_dem_JOE', 'pri_dem_JOHN',
       'pri_dem_MIKE', 'pri_rep_DUNCAN', 'pri_rep_FRED', 'pri_rep_JOHN',
       'pri_rep_MIKE', 'pri_rep_MITT', 'pri_rep_RON', 'pri_rep_RUDY',
       'pri_rep_TOM', 'gen_dem_JOE', 'gen_ind_DARRELL', 'gen_ind_DAVID',
       'gen_ind_MATT', 'gen_ind_ROBERT', 'gen_ind_STEWART', 'gen_lib_WAYNE',
       'gen_rep_SARAH', 'gen_wgr_ROSA', 'rep_primary_total',
       'dem_primary_total', 'rep_general_total', 'dem_general_total',
       'lib_general_total', 'wgr_general_total', 'ind_general_total'],
      dtype='object')

Now, we save the cleaned dataframe into the processed directory.

In [76]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "WI.csv", index=False)